In [28]:
import pandas as pd
import numpy as np

In [29]:
path=r"C:\Users\PC\OneDrive\Desktop\EDHEC Buisness School\MASI 16.xlsx"
df=pd.read_excel(path,sheet_name='Sheet1',header=0,index_col=0,parse_dates=True)
df.columns = df.columns.str.strip()
df

C:\Users\PC\AppData\Local\Temp\ipykernel_274300\2008801284.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df=pd.read_excel(path,sheet_name='Sheet1',header=0,index_col=0,parse_dates=True)


,Date,Assets,Return,Market
Number,,,,
1,2021-01-01,ADH,0.1080,0.0339
2,2021-02-01,ADH,-0.0960,-0.0323
3,2021-03-01,ADH,-0.0328,0.0064
4,2021-04-01,ADH,0.0792,0.0279
5,2021-05-01,ADH,0.2006,0.0424
...,...,...,...,...
764,2024-08-01,TQM,-0.0444,-0.0073
765,2024-09-01,TQM,0.1240,0.0432
766,2024-10-01,TQM,0.0110,-0.0064


In [30]:
df = df.reset_index()  # now 'Date' is only a column
df.columns=df.columns.str.strip()

## Low volatility 

In [31]:
df["Vol_12m"] = (
    df
    .groupby("Assets")["Return"]
    .rolling(window=12)
    .std()
    .reset_index(level=0, drop=True))

In [32]:
df.groupby('Assets')['Vol_12m'].std().describe()


count    16.000000
mean      0.023960
std       0.026846
min       0.005765
25%       0.010869
50%       0.015251
75%       0.022856
max       0.117149
Name: Vol_12m, dtype: float64

In [33]:
df.groupby('Date')['Vol_12m'].std().describe()


count    37.000000
mean      0.050176
std       0.025216
min       0.016321
25%       0.032661
50%       0.040677
75%       0.049625
max       0.095147
Name: Vol_12m, dtype: float64

In [34]:
df['Vol_rank'] = df.groupby('Date')['Vol_12m'].rank(pct=True)

In [35]:
def low_vol_factor(df, high_q=0.7, low_q=0.3):

    factor_returns = []

    for date, group in df.groupby('Date'):

        group = group.dropna(subset=['Vol_12m', 'Return'])

        if len(group) < 6:
            continue

        low_vol  = group[group['Vol_rank'] <= low_q]
        high_vol = group[group['Vol_rank'] >= high_q]

        if low_vol.empty or high_vol.empty:
            continue

        factor_returns.append({
            'Date': date,
            'LOWVOL_Factor': low_vol['Return'].mean() - high_vol['Return'].mean()
        })

    factor_df = pd.DataFrame(factor_returns)

    if factor_df.empty:
        raise ValueError("Low-Vol factor construction failed")

    return factor_df.set_index('Date').sort_index()


In [36]:
lowvol = low_vol_factor(df)
df = df.merge(
    lowvol,
    left_on='Date',
    right_index=True,
    how='left'
)
df

,Number,Date,Assets,Return,Market,Vol_12m,Vol_rank,LOWVOL_Factor
0,1,2021-01-01,ADH,0.1080,0.0339,NaN,NaN,NaN
1,2,2021-02-01,ADH,-0.0960,-0.0323,NaN,NaN,NaN
2,3,2021-03-01,ADH,-0.0328,0.0064,NaN,NaN,NaN
3,4,2021-04-01,ADH,0.0792,0.0279,NaN,NaN,NaN
4,5,2021-05-01,ADH,0.2006,0.0424,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
763,764,2024-08-01,TQM,-0.0444,-0.0073,0.056473,0.5625,-0.009945
764,765,2024-09-01,TQM,0.1240,0.0432,0.057545,0.5625,-0.147700
765,766,2024-10-01,TQM,0.0110,-0.0064,0.057677,0.6250,-0.078130
766,767,2024-11-01,TQM,-0.1023,0.0499,0.063927,0.6875,-0.087795


In [37]:
df=df.dropna(subset=['Vol_12m','Return','LOWVOL_Factor'])

In [38]:
import statsmodels.api as sm
results = []

for asset in df['Assets'].unique():
    df_a = df[df['Assets'] == asset]
    y = df_a['Return']
    X = df_a[['Market','LOWVOL_Factor']]
    X = sm.add_constant(X)  # adds alpha
    
    model = sm.OLS(y, X).fit()
    
    results.append({
        'Asset': asset,
        'Alpha': model.params['const'],
        'Beta_Market': model.params['Market'],
        'Beta_LOWVOL': model.params['LOWVOL_Factor'],
        'R_squared': model.rsquared
    })

regression_2f = pd.DataFrame(results)

In [39]:
regression_2f.sort_values(by='R_squared', ascending=False)

,Asset,Alpha,Beta_Market,Beta_LOWVOL,R_squared
13,R DAR SAADA,-0.063227,0.694634,-2.054537,0.819978
2,ATW,0.003406,1.077845,0.094559,0.721470
0,ADH,-0.014751,1.073071,-1.100560,0.692713
7,CSR,-0.010251,1.018135,0.085617,0.590714
3,BCP,0.000618,0.947029,0.121151,0.535291
6,CMA,-0.004767,1.080079,-0.011733,0.521424
11,LHM,-0.007097,0.999960,0.010976,0.461277
12,MARSA,0.014065,1.261854,0.037187,0.458917
4,BOA,0.001789,0.786310,0.040787,0.450283
8,IAM,-0.013794,0.980390,0.114924,0.418030


## Value Factor 

In [40]:
path2=r"C:\Users\PC\OneDrive\Desktop\EDHEC Buisness School\MASI 16.xlsx"
df_2=pd.read_excel(path2,sheet_name='Sheet2',header=0,index_col=0,parse_dates=True)
df_2.columns = df_2.columns.str.strip()

C:\Users\PC\AppData\Local\Temp\ipykernel_274300\2143567247.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_2=pd.read_excel(path2,sheet_name='Sheet2',header=0,index_col=0,parse_dates=True)


In [41]:
df_2

,Date,Assets,Return,Market,LOWVOL_Factor,Value,Quality
Number,,,,,,,
1,2022-01-01,ADH,0.0980,0.0475,-0.082795,2.534294,-0.009545
2,2022-02-01,ADH,-0.0215,-0.0473,-0.119735,2.534294,-0.009545
3,2022-03-01,ADH,-0.3640,-0.0309,0.068480,2.534294,-0.009545
4,2022-04-01,ADH,0.1172,0.0333,-0.070040,2.534294,-0.009545
5,2022-05-01,ADH,-0.0370,-0.0639,-0.006415,2.534294,-0.009545
...,...,...,...,...,...,...,...
572,2024-08-01,TQM,-0.0444,-0.0073,-0.009945,0.270000,0.046629
573,2024-09-01,TQM,0.1240,0.0432,-0.147700,0.270000,0.046629
574,2024-10-01,TQM,0.0110,-0.0064,-0.078130,0.270000,0.046629


In [42]:
# First, fix the Value_Portfolio column by extracting the appropriate value
# Assuming each row should get one value, we extract the first element if it's a list
# Recompute Value_Portfolio correctly per Date and then compute Value_Factor
def _assign_value_portfolio_series(s):
    q_low = s.quantile(0.3)
    q_high = s.quantile(0.7)
    return pd.cut(s, bins=[-np.inf, q_low, q_high, np.inf], labels=['Low', 'Middle', 'High'])

df_2['Value_Portfolio'] = df_2.groupby('Date')['Value'].transform(_assign_value_portfolio_series).astype(str)

value_factor = (
    df_2[df_2['Value_Portfolio'].isin(['High', 'Low'])]
    .groupby(['Date', 'Value_Portfolio'])['Return']
    .mean()
    .unstack()
)

value_factor['Value_Factor'] = value_factor.get('High') - value_factor.get('Low')
value_factor = value_factor[['Value_Factor']]
df_2 = df_2.merge(value_factor, on='Date', how='left')

In [43]:
results = []

for asset in df_2['Assets'].unique():
    df_a = df_2[df_2['Assets'] == asset]
    y = df_a['Return']
    X = df_a[['Market','Value_Factor','LOWVOL_Factor']]
    X = sm.add_constant(X)  # adds alpha
    
    model = sm.OLS(y, X).fit()
    
    results.append({
        'Asset': asset,
        'Alpha': model.params['const'],
        'Beta_Market': model.params['Market'],
        'Beta_Value': model.params['Value_Factor'],
        'Beta_LOWVOL': model.params['LOWVOL_Factor'],
        'R_squared': model.rsquared
    })

regression_3f = pd.DataFrame(results)

In [44]:
regression_3f.sort_values(by='R_squared', ascending=False)

,Asset,Alpha,Beta_Market,Beta_Value,Beta_LOWVOL,R_squared
13,R DAR SAADA,-0.066620,0.724311,0.575102,-1.691769,0.832832
0,ADH,-0.017674,1.127162,0.741893,-0.621167,0.741115
2,ATW,0.003259,1.080848,0.039685,0.120282,0.721159
7,CSR,-0.009498,1.013759,-0.108460,0.018084,0.604851
3,BCP,0.000903,0.979244,0.251773,0.293849,0.591017
6,CMA,-0.007364,1.054046,0.018905,-0.019313,0.534816
12,MARSA,0.014115,1.231466,-0.267422,-0.143480,0.487697
4,BOA,0.001654,0.805131,0.175359,0.158459,0.481184
11,LHM,-0.008464,1.002283,0.148448,0.100760,0.476677
15,TQM,-0.000644,1.006075,-0.129875,-0.137745,0.431678


## Quality Factor 

In [45]:
def assign_quality_portfolio_series(s):
    q_low = s.quantile(0.3)
    q_high = s.quantile(0.7)
    return pd.cut(s, bins=[-np.inf, q_low, q_high, np.inf], labels=['Low', 'Middle', 'High']).astype(str)

# Apply to df_2 which contains the 'Quality' column
df_2['Quality_Portfolio'] = df_2.groupby('Date')['Quality'].transform(assign_quality_portfolio_series)

quality_factor = (
    df_2[df_2['Quality_Portfolio'] != 'Middle']
    .groupby(['Date', 'Quality_Portfolio'])['Return']
    .mean()
    .unstack()
)

quality_factor['Quality_Factor'] = quality_factor['High'] - quality_factor['Low']
quality_factor = quality_factor[['Quality_Factor']]

df_2 = df_2.merge(
    quality_factor,
    left_on='Date',
    right_index=True,
    how='left'
)

In [46]:
df_2.groupby('Assets')['Quality_Factor'].nunique()

Assets
ADH            36
ALLIANCE       36
ATW            36
BCP            36
BOA            36
CDM            36
CMA            36
CSR            36
IAM            36
JET            36
LHM            36
Label V        36
MARSA          36
R DAR SAADA    36
SONASID        36
TQM            36
Name: Quality_Factor, dtype: int64

In [47]:
results = []

for asset in df_2['Assets'].unique():
    df_a = df_2[df_2['Assets'] == asset]
    y = df_a['Return']
    X = df_a[['Market','Value_Factor','LOWVOL_Factor', 'Quality_Factor']]
    X = sm.add_constant(X)  # adds alpha
    
    model = sm.OLS(y, X).fit()
    
    results.append({
        'Asset': asset,
        'Alpha': model.params['const'],
        'Beta_Market': model.params['Market'],
        'Beta_Value': model.params['Value_Factor'],
        'Beta_LOWVOL': model.params['LOWVOL_Factor'],
        'Beta_Quality': model.params['Quality_Factor'],
        'R_squared': model.rsquared
    })

regression_4f = pd.DataFrame(results)

In [48]:
df_2[['Market', 'Value_Factor', 'Quality_Factor', 'LOWVOL_Factor']].corr()


,Market,Value_Factor,Quality_Factor,LOWVOL_Factor
Market,1.000000,0.228309,-0.180833,-0.333152
Value_Factor,0.228309,1.000000,-0.866767,-0.835434
Quality_Factor,-0.180833,-0.866767,1.000000,0.857827
LOWVOL_Factor,-0.333152,-0.835434,0.857827,1.000000


In [49]:
regression_4f.sort_values(by='R_squared', ascending=False)

,Asset,Alpha,Beta_Market,Beta_Value,Beta_LOWVOL,Beta_Quality,R_squared
13,R DAR SAADA,-0.043398,1.054503,-0.248326,-1.031467,-1.910754,0.891298
2,ATW,0.006663,1.129238,-0.080990,0.217051,-0.280025,0.756816
0,ADH,-0.015058,1.164349,0.649155,-0.546802,-0.215196,0.742861
3,BCP,0.005445,1.043829,0.090712,0.423004,-0.373741,0.652283
7,CSR,-0.011720,0.982165,-0.029672,-0.045096,0.182826,0.618673
12,MARSA,0.007319,1.134821,-0.026410,-0.336746,0.559264,0.549763
6,CMA,-0.006509,1.066204,-0.011413,0.004999,-0.070353,0.536299
11,LHM,-0.013536,0.930164,0.328299,-0.043461,0.417342,0.531344
4,BOA,0.004154,0.840691,0.086680,0.229570,-0.205779,0.503066
8,IAM,-0.018385,0.901480,0.075999,-0.095839,0.417338,0.481598


## Return Factor

In [50]:
from functools import reduce

# compare R_squared across 2-factor, 3-factor and 4-factor regressions
r2_2 = regression_2f[['Asset', 'R_squared']].rename(columns={'R_squared': 'M & LV'})
r2_3 = regression_3f[['Asset', 'R_squared']].rename(columns={'R_squared': 'M & LV & V'})
r2_4 = regression_4f[['Asset', 'R_squared']].rename(columns={'R_squared': 'M & V & LV & Q'})

comp = reduce(lambda L, R: pd.merge(L, R, on='Asset', how='outer'), [r2_2, r2_3, r2_4])
comp[['M & LV','M & LV & V','M & V & LV & Q']] = comp[['M & LV','M & LV & V','M & V & LV & Q']].astype(float)
comp = comp.set_index('Asset').sort_values('M & V & LV & Q', ascending=False)
comp

,M & LV,M & LV & V,M & V & LV & Q
Asset,,,
R DAR SAADA,0.819978,0.832832,0.891298
ATW,0.721470,0.721159,0.756816
ADH,0.692713,0.741115,0.742861
BCP,0.535291,0.591017,0.652283
CSR,0.590714,0.604851,0.618673
MARSA,0.458917,0.487697,0.549763
CMA,0.521424,0.534816,0.536299
LHM,0.461277,0.476677,0.531344
BOA,0.450283,0.481184,0.503066


In [51]:
comp['3_vs_2'] = comp['M & LV & V']  - comp['M & LV']
comp['4_vs_3'] = comp['M & V & LV & Q'] - comp['M & LV & V']

In [52]:
comp

,M & LV,M & LV & V,M & V & LV & Q,3_vs_2,4_vs_3
Asset,,,,,
R DAR SAADA,0.819978,0.832832,0.891298,0.012855,0.058465
ATW,0.721470,0.721159,0.756816,-0.000310,0.035657
ADH,0.692713,0.741115,0.742861,0.048402,0.001747
BCP,0.535291,0.591017,0.652283,0.055726,0.061267
CSR,0.590714,0.604851,0.618673,0.014136,0.013822
MARSA,0.458917,0.487697,0.549763,0.028779,0.062066
CMA,0.521424,0.534816,0.536299,0.013391,0.001483
LHM,0.461277,0.476677,0.531344,0.015400,0.054666
BOA,0.450283,0.481184,0.503066,0.030900,0.021883


In [53]:
comp['3_vs_2'].mean()

np.float64(0.028546553506288375)

In [54]:
comp['4_vs_3'].mean()

np.float64(0.03487768519271627)